[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/csg_raytracer/csg_raytracer.ipynb)

# Constructive Solid Geometry and Recursive Ray Tracing

This example renders a constructive solid geometry scene in 3D projective geometric algebra (PGA), running offscreen on the GPU through WebGPU compute shaders. The scene pairs a singly ruled hyperboloid, an open hollow bowl formed by boolean subtraction and plane clipping, and a polished mirror sphere reflecting the surrounding geometry, all resting on a checkered floor with mutual drop shadows.

A solid quadric is a symmetric bilinear map from points to planes, `Plane <- Point`, vanishing on its boundary and negative inside. Bound to a ray, it produces a scalar quadratic in ray distance whose roots determine entry and exit intervals. Hyperboloids of one sheet possess signature (+, +, -, -) and carry two infinite families of straight generator lines, each lying entirely within the curved surface.

In the matrix formulation of projective geometry, evaluating a quadric $Q$ at a point $\mathbf x$ reads as the quadratic form $\mathbf x^\top Q \mathbf x = 0$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from examples.animation import save_figure, show_animation
from examples.quadrics.csg_raytracer import core, render, scenarios

np.set_printoptions(precision=4, suppress=True)

## 1. The Ruled Hyperboloid and Straight Generator Lines

A ruled hyperboloid of one sheet is constructed by taking coordinate planes $x$ and $y$ with positive metric, $z$ with negative metric, and the plane at infinity $w$. Through every point on the throat circle pass two straight lines that lie entirely within the surface.

In [ ]:
# Construct the hyperboloid of one sheet with semi-axes a = b = 0.55, c = 0.85
hyp = core.hyperboloid([0.55, 0.55, 0.85])

# Compute straight generator lines along the throat circle
n_rulings = 16
throat_points, v_plus, v_minus = core.ruling_lines([0.55, 0.55, 0.85], n_rulings)

# Verify that all generator lines lie identically on the surface for all distances t
t_samples = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
for t in t_samples:
    pts_on_line = throat_points + v_plus * t
    values = core.evaluate(hyp, pts_on_line)
    assert np.allclose(values.to_array(), 0.0, atol=1e-7)
print("Generator lines verified: quadric form vanishes identically along every ruling.")

## 2. Hollow Clipped Bowl by Boolean CSG

A hollow bowl is formed by taking an outer solid ellipsoid, subtracting a concentric inner cavity ellipsoid (leaving a thin structural shell), and clipping the upper hemisphere with a tilted cutting plane.

In [ ]:
# Outer ellipsoid and inner hollow cavity (20% wall thickness)
r_out = np.array([0.75, 0.75, 0.75])
r_in = r_out * 0.80

bowl_out = core.ellipsoid(r_out)
bowl_in = core.ellipsoid(r_in)

# Place the bowl with a motor that translates and tilts it forward
motor_bowl = core.place(bowl_out, scenarios.BOWL_POSITION, scenarios.BOWL_TILT_YZ, turn_axis="yz")
print("Hollow bowl primitives constructed.")

## 3. Assembling the Scene and GPU Ray Tracing

The complete scene includes the ruled hyperboloid, the hollow clipped bowl, a chrome mirror sphere reflecting the environment, and a checkered ground plane. The shader is transpiled directly from GA expressions to WGSL and executed on the GPU via WebGPU compute shaders.

In [ ]:
# Build the complete scene dictionary of algebraic elements
scene = scenarios.make_scene()

# Render the showcase image using the GPU compute pipeline
fig = render.draw_scene(scene, scenarios.PRIMARY_CAM, scenarios.TARGET, 640, 640)
save_figure(fig, "csg_raytracer")

## 4. Turntable Animation

An orbit camera circles the scene over 48 frames, capturing the changing reflections on the mirror sphere, the dynamic shadows, and the interior cavity of the clipped bowl.

In [ ]:
# Generate camera orbit trajectory and render animation
n_frames = 48
camera_orbit = scenarios.orbit_cameras(n_frames)
anim_frames = render.frames(scene, camera_orbit, scenarios.TARGET, 480, 480)
show_animation(anim_frames, "csg_raytracer", scenarios.FRAME_DURATION)